In [ ]:
import dotenv
import os
dotenv.load_dotenv("../env_workshop")

# Scoping - User Clarification & Brief Generation

**User Clarification** *Every research starts with a user inquiry. The goal of the scoping is to collect as much additional information from the user as needed for the research*

**Brief Generation** *When all the relevant information is gathered, a brief is generated to perform the research.*

Here is our overall research flow:

![alt text](image-1.png)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from utils import show_prompt

In [ ]:
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")
PHOENIX_PROJECT_NAME=os.environ.get("PHOENIX_PROJECT_NAME")

In [ ]:
from phoenix.otel import register
from openinference.instrumentation import using_metadata
from openinference.instrumentation.langchain import LangChainInstrumentor
from opentelemetry import trace

In [ ]:
if os.environ.get("PHOENIX_COLLECTOR_ENDPOINT"):
    # configure the Phoenix tracer
    tracer_provider = register(
        project_name=PHOENIX_PROJECT_NAME, 
        auto_instrument=False 
    )
    
else:
    tracer_provider = trace.NoOpTracerProvider()
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = trace.get_tracer(__name__)

### LLM Prompt for iteratively interacting with the user and ask clarifying questions.

In [ ]:
clarify_with_user_instructions = """You are an expert research assistant. You will be given the conversation so far between the user and assistant:
<Messages>
{messages}
</Messages>

Date: {date}

Task: Decide whether you need additional information to start research. Be conservative: only ask if absolutely necessary. Before asking a new question, check if that question was already answered or not. If not, only then re-ask that question.

Respond in valid JSON format with these exact keys:
"need_clarification": boolean,
"question": "<one concise, single multipart question using a short bullet list to collect all missing items>",
"verification": "<one-line acknowledgement summarizing the user's request and confirming you will start research>"

If you need to ask a clarifying question, return:
"need_clarification": true,
"question": "<your clarifying question>",
"verification": ""

If you do not need to ask a clarifying question, return:
"need_clarification": false,
"question": "",
"verification": "<acknowledgement message that you will now start research based on the provided information>"

If you ask a question, collect all missing high-level items in one or more messages using this short bullet list format (use markdown bullets so it renders nicely if viewed in markdown):
- Goal: (what should the research achieve?)
- Scope: (topics, geography, timeframe)
- Success criteria: (what will make the research useful?)
- Constraints: (budget, languages, sources to include/avoid, deadlines)
- Preferred sources or formats: (links, types of sources, output format)

Do not include anything else outside the required JSON object."""

In [ ]:
show_prompt(clarify_with_user_instructions,"clarify_with_user_instructions")

### State

The state object serves as our primary mechanism for storing and passing context between different phases of the research workflow. 

We can use it to [write and select context](https://blog.langchain.com/context-engineering-for-agents/) that will be used to guide the research.


In [ ]:
"""State Definitions and Pydantic Schemas for Research Scoping.

This defines the state objects and structured schemas used for
the research agent scoping workflow, including researcher state management and output schemas.
"""

import operator
from typing_extensions import Optional, Annotated, List, Sequence

from langchain_core.messages import BaseMessage
from langgraph.graph import MessagesState
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

# ===== STATE DEFINITIONS =====

class AgentInputState(MessagesState):
    """Input state for the full agent - only contains messages from user input."""
    pass

class AgentState(MessagesState):
    """
    Main state for the full multi-agent research system.
    
    Extends MessagesState with additional fields for research coordination.
    Note: Some fields are duplicated across different state classes for proper
    state management between subgraphs and the main workflow.
    """

    # Research brief generated from user conversation history
    research_brief: Optional[str]
    # Messages exchanged with the supervisor agent for coordination
    supervisor_messages: Annotated[Sequence[BaseMessage], add_messages]
    # Raw unprocessed research notes collected during the research phase
    raw_notes: Annotated[list[str], operator.add] = []
    # Processed and structured notes ready for report generation
    notes: Annotated[list[str], operator.add] = []
    # Final formatted research report
    final_report: str

# ===== STRUCTURED OUTPUT SCHEMAS =====

class ClarifyWithUser(BaseModel):
    """Schema for user clarification decision and questions."""
    
    need_clarification: bool = Field(
        description="Whether the user needs to be asked a clarifying question.",
    )
    question: str = Field(
        description="A question to ask the user to clarify the report scope",
    )
    verification: str = Field(
        description="Verify message that we will start research after the user has provided the necessary information.",
    )

class ResearchQuestion(BaseModel):
    """Schema for structured research brief generation."""
    
    research_brief: str = Field(
        description="A research question that will be used to guide the research.",
    )

In [ ]:
transform_messages_into_research_topic_prompt = """You are an expert research assistant. You will be given the conversation so far between you and the user:
<Messages>
{messages}
</Messages>

Date: {date}

Task: Produce a single, concise research brief (few short sentences, first person) that will be used to guide research. The brief must:
- Restate the user's explicit request and any stated preferences/constraints.
- If important dimensions are missing, list them under "Open considerations:" as short bullet points (do NOT assume values).
- Mention any user-specified preferred sources or output format if present.
- Never invent facts, preferences, or constraints that the user did not state.

Output: A single short paragraph (or the user's brief) suitable for the researcher to act on. Do not output JSON or extra metadata—only the research brief text."""

In [ ]:
"""User Clarification and Research Brief Generation.

This module implements the scoping phase of the research workflow, where we:
1. Assess if the user's request needs clarification
2. Generate a detailed research brief from the conversation

The workflow uses structured output to make deterministic decisions about whether sufficient context exists to proceed with research.
"""

from datetime import datetime
from typing_extensions import Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, get_buffer_string
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_openai import ChatOpenAI


# ===== UTILITY FUNCTIONS =====

def get_today_str() -> str:
    """Get current date in a human-readable format."""
    return datetime.now().strftime("%a %b %-d, %Y")

# ===== CONFIGURATION =====

# Initialize model
model = ChatOpenAI(
    model="gpt-4.1",
    base_url=OPENAI_BASE_URL,
    temperature=0.0,
    #max_tokens=512,
)

# ===== WORKFLOW NODES =====

def clarify_with_user(state: AgentState) -> Command[Literal["write_research_brief", "__end__"]]:
    """
    Determine if the user's request contains sufficient information to proceed with research.
    
    Uses structured output to make deterministic decisions and avoid hallucination.
    Routes to either research brief generation or ends with a clarification question.
    """
    # Set up structured output model
    structured_output_model = model.with_structured_output(ClarifyWithUser)

    # Invoke the model with clarification instructions
    response = structured_output_model.invoke([
        HumanMessage(content=clarify_with_user_instructions.format(
            messages=get_buffer_string(messages=state["messages"]), 
            date=get_today_str()
        ))
    ])
    
    # Route based on clarification need
    if response.need_clarification:
        return Command(
            goto=END, 
            update={"messages": [AIMessage(content=response.question)]}
        )
    else:
        return Command(
            goto="write_research_brief", 
            update={"messages": [AIMessage(content=response.verification)]}
        )

def write_research_brief(state: AgentState):
    """
    Transform the conversation history into a comprehensive research brief.
    
    Uses structured output to ensure the brief follows the required format
    and contains all necessary details for effective research.
    """
    # Set up structured output model
    structured_output_model = model.with_structured_output(ResearchQuestion)
    
    # Generate research brief from conversation history
    response = structured_output_model.invoke([
        HumanMessage(content=transform_messages_into_research_topic_prompt.format(
            messages=get_buffer_string(state.get("messages", [])),
            date=get_today_str()
        ))
    ])
    
    # Update state with generated research brief and pass it to the supervisor
    return {
        "research_brief": response.research_brief,
        "supervisor_messages": [HumanMessage(content=f"{response.research_brief}.")]
    }

# ===== GRAPH CONSTRUCTION =====

# Build the scoping workflow
deep_researcher_builder = StateGraph(AgentState, input_schema=AgentInputState)

# Add workflow nodes
deep_researcher_builder.add_node("clarify_with_user", clarify_with_user)
deep_researcher_builder.add_node("write_research_brief", write_research_brief)

# Add workflow edges
deep_researcher_builder.add_edge(START, "clarify_with_user")
deep_researcher_builder.add_edge("write_research_brief", END)

# Compile the workflow
scope_research = deep_researcher_builder.compile()

In [ ]:
# Compile with in-memory checkpointer to test in notebook
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
scope = deep_researcher_builder.compile(checkpointer=checkpointer)
display(Image(scope.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# Run the workflow
from utils import format_messages
from langchain_core.messages import HumanMessage
thread = {"configurable": {"thread_id": "1"}}
result = scope.invoke({"messages": [HumanMessage(content="What is the best coffee shop in seattle.")]}, config=thread)
format_messages(result['messages'])

In [ ]:
human_answer = "Quality of the coffee bean and the atmosphere of the shop are the two most important factors for me."
result = scope.invoke({"messages": [HumanMessage(content=human_answer)]}, config=thread)
format_messages(result['messages'])

In [ ]:
human_answer = "I am looking for best coffee shop in and around Seattle. I am interested in independent coffee shops, but chains are acceptable."
result = scope.invoke({"messages": [HumanMessage(content=human_answer)]}, config=thread)
format_messages(result['messages'])

In [ ]:
from rich.markdown import Markdown
Markdown(result["research_brief"])

In [ ]:
result